In [20]:
import logging
from typing import Dict, List, Callable, Any, Optional, Type
from types import SimpleNamespace
from semantic_kernel import Kernel
from semantic_kernel.functions import KernelFunction
from semantic_kernel.agents.azure_ai.azure_ai_agent import AzureAIAgent, AzureAIAgentThread
import inspect
# Import all specialized agent implementations
from semantic_kernel.prompt_template.prompt_template_config import PromptTemplateConfig

from azure.ai.projects.models import (
    ResponseFormatJsonSchema,
    ResponseFormatJsonSchemaType)

In [2]:
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, ClientSecretCredential
from azure.ai.projects.aio import AIProjectClient
import os

In [21]:
load_dotenv("../backend/.env")

True

In [22]:
try:
    credential = DefaultAzureCredential()
    if credential is None:
        raise RuntimeError(
            "Unable to acquire Azure credentials; ensure DefaultAzureCredential is configured"
        )

    connection_string = os.getenv("AZURE_AI_AGENT_PROJECT_CONNECTION_STRING")
    ai_project_client = AIProjectClient.from_connection_string(
        credential=credential, conn_str=connection_string
    )
except Exception as exc:
    logging.error("Failed to create AIProjectClient: %s", exc)
    raise


In [23]:
agents = await ai_project_client.agents.list_agents()

In [25]:
found_agent = False
for agent in agents.data:
    if agent.name == "Enterprise_Agent":
        found_agent = agent
        
        break

In [35]:
found_agent.id

'asst_7lS1oHPaebUjlv02zB08eM77'

In [39]:
country_name = "Cuba"
# Create AzureAIAgent instance
azure_agent = AzureAIAgent(
    client=ai_project_client,
    definition=found_agent
)

# Create thread using AzureAIAgentThread
agent_thread = AzureAIAgentThread(client=ai_project_client)

# Invoke the agent
response_content = ""
# async for response_item in azure_agent.invoke(
#     thread=agent_thread,
#     messages=f"Get the internal risk details for {country_name}"
# ):
#     if response_item and response_item.message:
#         response_content += str(response_item.message.content)
#         print(f"Agent: {response_item.message.content}")

# print(f"\nFull response: {response_content}")

async_generator = azure_agent.invoke(
                messages=f"Get the internal risk details for {country_name}",
                thread=agent_thread,
            )
response_content = ""

# Collect the response from the async generator
async for chunk in async_generator:
    if chunk is not None:
        response_content += str(chunk)
        print(f"Agent: {chunk}")

print(f"\nFull response: {response_content}")

Agent: #### Cuba Risk and Sanctions Information

##### Risk Category
- **Risk Level:** Very High
- **Sanctions Status:** Yes

##### Sources
- [List of All Countries In The World]: Internal document ID 【3:0†source】

Full response: #### Cuba Risk and Sanctions Information

##### Risk Category
- **Risk Level:** Very High
- **Sanctions Status:** Yes

##### Sources
- [List of All Countries In The World]: Internal document ID 【3:0†source】


In [37]:
country_name = "Cuba"
thread = await ai_project_client.agents.create_thread()
message = await ai_project_client.agents.create_message(
    thread_id=thread.id,
    content=f"Get the internal risk details {country_name}",
    role="user",
)
run = await ai_project_client.agents.create_and_process_run( thread_id=thread.id, agent_id=found_agent.id)
if run.status == "failed":
    print(f"Run failed : {run.last_error}")

messages = await ai_project_client.agents.list_messages(thread_id=thread.id)
for message in messages.data:
    print(f"Role: {message.role}, Content: {message.content}")

last_msg = messages.get_last_text_message_by_role(role="assistant")
print('Agent:', last_msg.text.value, '\n\n')
print(last_msg)


Role: MessageRole.AGENT, Content: [{'type': 'text', 'text': {'value': '#### Cuba Risk and Sanctions Information\n\n##### Risk Category\n- **Risk Level:** Very High\n- **Sanctions Status:** Yes\n\n##### Sources\n- [List of All Countries In The World]: Internal document ID 【3:0†source】', 'annotations': [{'type': 'url_citation', 'text': '【3:0†source】', 'start_index': 194, 'end_index': 206, 'url_citation': {'url': 'doc_0', 'title': 'sanctionslist.xlsx'}}]}}]
Role: MessageRole.USER, Content: [{'type': 'text', 'text': {'value': 'Get the internal risk details Cuba', 'annotations': []}}]
Agent: #### Cuba Risk and Sanctions Information

##### Risk Category
- **Risk Level:** Very High
- **Sanctions Status:** Yes

##### Sources
- [List of All Countries In The World]: Internal document ID 【3:0†source】 


{'type': 'text', 'text': {'value': '#### Cuba Risk and Sanctions Information\n\n##### Risk Category\n- **Risk Level:** Very High\n- **Sanctions Status:** Yes\n\n##### Sources\n- [List of All Count

In [1]:
for agent in agents.data:
    print(f"Agent ID: {agent.id} Name: {agent.name} Description: {agent.description}")
    if agent.id == "asst_lUMOIEt2p0qY3pQgE2WHljCf":
        print(f"Deleting agent {agent.id}...")
        # Delete the agent
        status = await ai_project_client.agents.delete_agent("asst_lUMOIEt2p0qY3pQgE2WHljCf")
        break
    

NameError: name 'agents' is not defined